# Model provenance and citations

This notebook fine-tunes Qwen3-4B (Qwen Team, 2025) for the ClimateAdapt Planning Support Agent. Parameter-efficient adaptation is based on Low-Rank Adaptation (LoRA; Hu et al., 2022) and Quantized Low-Rank Adaptation (QLoRA; Dettmers et al., 2023). Training and GGUF export were implemented using the Unsloth framework (Unsloth AI, n.d.).

The project-specific contribution includes the ClimateAdapt instruction dataset, reporting schema, safety behaviour, human-review requirements, fine-tuning configuration, controlled evaluation and local deployment packaging.

- Base model: `Qwen/Qwen3-4B`
- Base-model licence: Apache License 2.0
- Fine-tuning method: QLoRA
- LoRA rank: 16
- LoRA alpha: 16
- Training framework: Unsloth
- Export format: GGUF
- Quantisation: Q4_K_M
- Ollama model name: `climateadapt-planner:4b`

Thinking mode was disabled during the controlled base-versus-fine-tuned evaluation so that both models used the same deterministic, schema-constrained inference conditions.

## References

Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). QLoRA: Efficient finetuning of quantized LLMs. *Advances in Neural Information Processing Systems, 36*. https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html

Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2022). LoRA: Low-rank adaptation of large language models. *International Conference on Learning Representations*. https://openreview.net/forum?id=nZeVKeeFYf9

Qwen Team. (2025). *Qwen3 technical report* [Preprint]. arXiv. https://doi.org/10.48550/arXiv.2505.09388

Unsloth AI. (n.d.). *Unsloth*. GitHub. Retrieved October 13, 2026, from https://github.com/unslothai/unsloth


# ClimateAdapt AI Ã¢â‚¬â€ Qwen3 4B QLoRA Fine-tuning

This notebook fine-tunes **Qwen3 4B** with **Unsloth 4-bit QLoRA** using only the ClimateAdapt training and validation splits. The held-out test split is never uploaded or opened.

Before running: select **Runtime Ã¢â€ â€™ Change runtime type Ã¢â€ â€™ T4 GPU**. Run the cells in order.

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo trl datasets accelerate bitsandbytes

In [ ]:
import json, os, platform, random, shutil, sys, time
from pathlib import Path

from unsloth import FastLanguageModel, is_bfloat16_supported
import numpy as np
import pandas as pd
import torch
import transformers
import trl
from datasets import load_dataset
from google.colab import drive, files

SEED = 8532
MODEL_NAME = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'
MAX_SEQUENCE_LENGTH = 2048
LOCAL_RUN_DIR = Path('/content/climateadapt_qwen3_4b_run')
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/ClimateAdapt_AI/qwen3_4b_qlora_v1')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), 'A Colab GPU is required. Select Runtime > Change runtime type > T4 GPU.'
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('TRL:', trl.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Base model:', MODEL_NAME)
print('Maximum sequence length:', MAX_SEQUENCE_LENGTH)

In [ ]:
drive.mount('/content/drive')
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Assessment artefacts will be copied to:', DRIVE_RUN_DIR)

## Upload the dataset

Select `climateadapt_qlora_train_validation.zip`. Do **not** upload the held-out test JSONL.

In [ ]:
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one training/validation zip file.'
uploaded_name = next(iter(uploaded))
assert uploaded_name.lower().endswith('.zip'), 'The uploaded file must be a ZIP archive.'

DATA_DIR = Path('/content/climateadapt_instruction_data')
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
shutil.unpack_archive(f'/content/{uploaded_name}', DATA_DIR)

train_candidates = list(DATA_DIR.rglob('climateadapt_instruction_train.jsonl'))
validation_candidates = list(DATA_DIR.rglob('climateadapt_instruction_validation.jsonl'))
test_candidates = list(DATA_DIR.rglob('*test*.jsonl'))
assert len(train_candidates) == 1, f'Expected one train JSONL, found {train_candidates}'
assert len(validation_candidates) == 1, f'Expected one validation JSONL, found {validation_candidates}'
assert not test_candidates, f'Held-out test data must not be uploaded: {test_candidates}'

TRAIN_FILE = train_candidates[0]
VALIDATION_FILE = validation_candidates[0]
print('Training file:', TRAIN_FILE)
print('Validation file:', VALIDATION_FILE)
print('Held-out test files found: 0')

In [ ]:
raw_dataset = load_dataset(
    'json',
    data_files={'train': str(TRAIN_FILE), 'validation': str(VALIDATION_FILE)},
)
assert len(raw_dataset['train']) == 192
assert len(raw_dataset['validation']) == 24

def to_prompt_completion(example):
    messages = example['messages']
    assert [message['role'] for message in messages] == ['system', 'user', 'assistant']
    return {
        'prompt': messages[:2],
        'completion': messages[2:],
    }

dataset = raw_dataset.map(
    to_prompt_completion,
    remove_columns=raw_dataset['train'].column_names,
    desc='Creating conversational prompt-completion records',
)
print(dataset)
print('Training records:', len(dataset['train']))
print('Validation records:', len(dataset['validation']))
print('Loss will be calculated on completion tokens only.')

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

def sequence_length(example):
    conversation = example['prompt'] + example['completion']
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return len(tokenizer(text, add_special_tokens=False)['input_ids'])

train_lengths = [sequence_length(example) for example in dataset['train']]
validation_lengths = [sequence_length(example) for example in dataset['validation']]
print('Training token length: min', min(train_lengths), 'mean', round(np.mean(train_lengths), 1), 'max', max(train_lengths))
print('Validation token length: min', min(validation_lengths), 'mean', round(np.mean(validation_lengths), 1), 'max', max(validation_lengths))
assert max(train_lengths + validation_lengths) <= MAX_SEQUENCE_LENGTH, 'Increase MAX_SEQUENCE_LENGTH to prevent truncation.'

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    output_dir=str(LOCAL_RUN_DIR / 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type='linear',
    optim='adamw_8bit',
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    max_length=MAX_SEQUENCE_LENGTH,
    packing=False,
    completion_only_loss=True,
    dataset_num_proc=2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=SEED,
    data_seed=SEED,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=training_config,
)
print('Trainer ready.')
print('Effective batch size:', training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps)

In [ ]:
started = time.perf_counter()
train_result = trainer.train()
training_seconds = time.perf_counter() - started
validation_metrics = trainer.evaluate()

print('Training completed in minutes:', round(training_seconds / 60, 2))
print('Final training loss:', train_result.metrics.get('train_loss'))
print('Best validation loss:', trainer.state.best_metric)
print('Final validation metrics:', validation_metrics)

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
history.to_csv(LOCAL_RUN_DIR / 'training_history.csv', index=False)

figure, axis = plt.subplots(figsize=(10, 6))
training_rows = history.dropna(subset=['loss']) if 'loss' in history else pd.DataFrame()
validation_rows = history.dropna(subset=['eval_loss']) if 'eval_loss' in history else pd.DataFrame()
if not training_rows.empty:
    axis.plot(training_rows['step'], training_rows['loss'], marker='o', label='Training loss')
if not validation_rows.empty:
    axis.plot(validation_rows['step'], validation_rows['eval_loss'], marker='s', label='Validation loss')
axis.set_title('ClimateAdapt Qwen3 4B QLoRA Training', weight='bold')
axis.set_xlabel('Optimizer step')
axis.set_ylabel('Completion-token loss')
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
figure.savefig(LOCAL_RUN_DIR / 'qwen3_4b_qlora_training_curves.png', dpi=180, bbox_inches='tight')
plt.show()

run_summary = {
    'experiment': 'ClimateAdapt Qwen3 4B QLoRA v1',
    'base_model': MODEL_NAME,
    'training_examples': len(dataset['train']),
    'validation_examples': len(dataset['validation']),
    'held_out_test_accessed': False,
    'seed': SEED,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'qlora_4bit': True,
    'lora_rank': 16,
    'lora_alpha': 16,
    'epochs': 3,
    'learning_rate': 1e-4,
    'effective_batch_size': 8,
    'completion_only_loss': True,
    'training_seconds': training_seconds,
    'train_metrics': train_result.metrics,
    'validation_metrics': validation_metrics,
    'best_validation_loss': trainer.state.best_metric,
    'gpu': torch.cuda.get_device_name(0),
    'versions': {
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'trl': trl.__version__,
    },
}
with open(LOCAL_RUN_DIR / 'fine_tuning_summary.json', 'w', encoding='utf-8') as output:
    json.dump(run_summary, output, ensure_ascii=False, indent=2, default=str)
print(json.dumps(run_summary, indent=2, default=str))

## Save the LoRA adapter and export GGUF

Q4_K_M is selected because it provides a practical quality/size balance for local Ollama deployment. Export may take several minutes.

In [ ]:
ADAPTER_DIR = LOCAL_RUN_DIR / 'climateadapt_qwen3_4b_lora_adapter'
GGUF_DIR = LOCAL_RUN_DIR / 'climateadapt_qwen3_4b_gguf'

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('LoRA adapter saved:', ADAPTER_DIR)

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method='q4_k_m',
)
gguf_files = list(GGUF_DIR.rglob('*.gguf'))
assert gguf_files, 'GGUF export did not produce a .gguf file.'
for path in gguf_files:
    print('GGUF:', path.name, round(path.stat().st_size / 1024**3, 3), 'GB')

In [ ]:
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(LOCAL_RUN_DIR, DRIVE_RUN_DIR)

print('=' * 72)
print('CLIMATEADAPT QLORA FINE-TUNING COMPLETED')
print('=' * 72)
print('Drive output:', DRIVE_RUN_DIR)
print('Adapter:', DRIVE_RUN_DIR / ADAPTER_DIR.name)
print('GGUF:', DRIVE_RUN_DIR / GGUF_DIR.name)
print('History:', DRIVE_RUN_DIR / 'training_history.csv')
print('Curves:', DRIVE_RUN_DIR / 'qwen3_4b_qlora_training_curves.png')
print('Summary:', DRIVE_RUN_DIR / 'fine_tuning_summary.json')
print('Held-out test dataset accessed: False')